In [1]:
import yaml
from app.datasets.loader import load_multiple_test_cases, load_test_cases
from app.datasets.validator import validate_dataset_schema
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.tests.nodes.reformulate import send_reformulate_requests, save_reformualte_responses, reformulate_tests

In [2]:
file_list = [
  './app/data/raw/rapido.xlsx',
]

test_config = {
  'GENERAL_TESTS': True,
  'TIMINGS': {'test': False, 'report': False},
  'TOKENS': {'test': True, 'report': False},
  'FOUNDRYS': {'test': False, 'report': False},
  'TRIAGE': {'test': False, 'report': False},
  'ROUTER': {'test': False, 'report': False},
  'GROUNDING': {'test': False, 'report': False},
  'SAVE_RESULTS': False,
  'PATH': './app/data/processed/reports/report_RAPIDO',
  
  'REFORMULATE': {'test': False, 'report': False}
}   

if file_list: 
  df = load_multiple_test_cases(file_list)
  df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)
test_timestamps = {}

In [ ]:
# TEST GENERALES
if test_config.get('GENERAL_TESTS', False):
  responses = client.query_batch(df['user_input'],df['reference'])
  save_responses_in_json, response_file_path = client.save_api_responses(responses)
  test_timestamps['general_tests'] = str(response_file_path).replace('\\', '/')

In [3]:
import json
import pandas as pd
import numpy as np

response_file_path = './app/data/processed/outcomes/outcome_20260507-130815.json'
with open(response_file_path, 'r', encoding='UTF-8') as f:
  responses = json.load(f)
test_timestamps['general_tests'] = 'outcome_20260507-130815.json'

In [4]:
if test_config.get('GENERAL_TESTS'):
  results, reports = run_tests(
    config = test_config, 
    data = responses, 
    df = df, 
    timestamp = test_timestamps
)

In [5]:
print(results)

{'timestamp': '20260507-130815', 'nodes': {}, 'tokens': {'in_ref': {'prom': 836.91, 'total': 83691, 'quantity': 100}, 'out_ref': {'prom': 25.8, 'total': 2580, 'quantity': 100}, 'in_tri': {'prom': 2648.36, 'total': 264836, 'quantity': 100}, 'out_tri': {'prom': 36.77, 'total': 3677, 'quantity': 100}, 'in_rou': {'prom': 0.0, 'total': 0, 'quantity': 0}, 'out_rou': {'prom': 0.0, 'total': 0, 'quantity': 0}, 'in_per': {'prom': 1139.683, 'total': 93454, 'quantity': 82}, 'out_per': {'prom': 138.488, 'total': 11356, 'quantity': 82}, 'in_gro': {'prom': 2809.732, 'total': 230398, 'quantity': 82}, 'out_gro': {'prom': 78.732, 'total': 6456, 'quantity': 82}, 'in_rag': {'prom': 2985.587, 'total': 223919, 'quantity': 75}, 'out_rag': {'prom': 143.56, 'total': 10767, 'quantity': 75}, 'retriever': {'prom': 15.667, 'total': 1175, 'quantity': 75}, 'in_tot': {'prom': 8962.98, 'total': 896298, 'quantity': 100}, 'out_tot': {'prom': 348.36, 'total': 34836, 'quantity': 100}}}
